# GENA-LM Sequence classification example
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AIRI-Institute/GENA_LM/blob/main/notebooks/GENA_sequence_classification_example.ipynb)

In [ ]:
!pip install torch --index-url https://download.pytorch.org/whl/cu118
!pip install transformers[torch] datasets
!pip install deepspeed

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu118

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import os
import shutil
import importlib
import numpy as np
import pandas as pd
import torch
import gc
import time
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModel, TrainingArguments, Trainer,
    DataCollatorWithPadding
)

MODEL_NAME = 'AIRI-Institute/gena-lm-bert-base-athaliana'
OUTPUT_BASE_DIR = "predictions_u300d300_L3"
MODEL_SAVE_DIR = "saved_models"
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def balance_dataset(df):
    class_0 = df[df['label'] == 0]
    class_1 = df[df['label'] == 1]
    n_samples = min(len(class_0), len(class_1))

    class_0 = resample(class_0, replace=False, n_samples=n_samples, random_state=1400)
    class_1 = resample(class_1, replace=False, n_samples=n_samples, random_state=1400)

    return pd.concat([class_0, class_1]).sample(frac=1, random_state=1400).reset_index(drop=True)

def prepare_dataset(df, seed_value):
    df = df[['sequence', 'peak', 'gene']].rename(columns={'sequence': 'text', 'peak': 'label'})

    # 第一次切分：Train 與 Temp (70% 用於驗證與測試)
    train_df, temp_df = train_test_split(
        df, test_size=0.7, random_state=seed_value, stratify=df['label']
    )

    # 第二次切分：從 Temp 中拆出 Val 與 Test
    val_df, test_df = train_test_split(
        temp_df, test_size=0.7, random_state=seed_value, stratify=temp_df['label']
    )

    return DatasetDict({
        'train': Dataset.from_pandas(balance_dataset(train_df)),
        'val': Dataset.from_pandas(balance_dataset(val_df)),
        'test': Dataset.from_pandas(balance_dataset(test_df))
    })

def preprocess_function(examples):
    """DNA 序列 Tokenization"""
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

def compute_metrics(eval_pred):
    """計算準確率"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = (predictions == labels).astype(np.float32).mean().item()
    return {'accuracy': acc}

def plot_and_save_metrics(true_labels, probs, predicted_labels, run_id):
    """繪製並儲存評估圖表"""
    # Confusion Matrix
    cm = confusion_matrix(true_labels, predicted_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues', values_format='d')
    plt.title(f'Confusion Matrix: Run {run_id}')
    plt.savefig(f"{OUTPUT_BASE_DIR}/confusion_matrix_run{run_id}.png")
    plt.close()

    # ROC AUC
    fpr, tpr, _ = roc_curve(true_labels, probs[:, 1])
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve: Run {run_id}')
    plt.legend(loc="lower right")
    plt.savefig(f"{OUTPUT_BASE_DIR}/roc_auc_run{run_id}.png")
    plt.close()

def run_single_experiment(df, run_id, base_seed):
    """執行單次訓練與評估"""
    current_seed = base_seed + run_id
    print(f"\n🚀 正在開始第 {run_id} 次實驗 (使用的 Seed: {current_seed})...")
    dataset = prepare_dataset(df, seed_value=current_seed)
    tokenized_dataset = dataset.map(preprocess_function, batched=True)

    base_model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
    module_path = base_model.__class__.__module__
    model_class = getattr(importlib.import_module(module_path), 'BertForSequenceClassification')
    model = model_class.from_pretrained(MODEL_NAME, num_labels=2)

    temp_output_dir = f"output_run_{run_id}"

    training_args = TrainingArguments(
        output_dir=temp_output_dir,
        learning_rate=5.5e-06,
        lr_scheduler_type="constant_with_warmup",
        warmup_ratio=0.1,
        optim='adamw_torch',
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=5,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        save_total_limit=1,
        seed=current_seed,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["val"],
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics
    )

    trainer.train()

    # 儲存預測與結果
    predictions = trainer.predict(tokenized_dataset['test'])
    logits = predictions.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    predicted_labels = np.argmax(logits, axis=1)
    true_labels = predictions.label_ids
    gene_names = tokenized_dataset['test']['gene']

    plot_and_save_metrics(true_labels, probs, predicted_labels, run_id)

    accuracy = compute_metrics((logits, true_labels))['accuracy']
    print(f"✅ Run {run_id} 準確率: {accuracy:.4f}")

    # 保存模型
    model.save_pretrained(f"{MODEL_SAVE_DIR}/best_u300d300_L3_model_run{run_id}")

    # 清理資源
    del model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    time.sleep(1)
    if os.path.exists(temp_output_dir):
        shutil.rmtree(temp_output_dir, ignore_errors=True)

    return accuracy

# --- 執行迴圈 ---
if __name__ == "__main__":
    try:
        df_all = pd.read_table("file/PN_sequence_up_300_downstream_300_seq_Nointron.txt")
        accuracies = []

        for i in range(1, 6):
            try:
                acc = run_single_experiment(df_all, run_id=i, base_seed=1400)
                accuracies.append(acc)
            except Exception as e:
                print(f"⚠️ 實驗 {i} 發生錯誤: {e}")
                continue

        if accuracies:
            print("\n" + "="*30)
            print(f"📊 平均準確率: {np.mean(accuracies):.4f}")
            print("="*30)
    except Exception as e:
        print(f"❌ 嚴重錯誤: {e}")

In [ ]:
import os
import torch
import pandas as pd
import importlib
from transformers import AutoModel, AutoTokenizer
from datasets import Dataset
from torch.nn.functional import softmax

# ==========================================
# 核心計算函數：計算序列區域重要性
# ==========================================
def calculate_importance(dataset, model, tokenizer, output_file):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    # 自動偵測特殊 Token，避免納入重要性計算
    special_token_set = {
        tokenizer.cls_token,
        tokenizer.sep_token,
        tokenizer.pad_token,
        tokenizer.eos_token,
        tokenizer.bos_token,
        None
    }

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, mode="w") as file:
        headers = ["Gene"] + [f"Region {i}" for i in range(1, 11)]
        file.write("\t".join(headers) + "\n")

        for sequence, gene_name in zip(dataset['text'], dataset['gene']):
            inputs = tokenizer(
                sequence,
                return_tensors="pt",
                truncation=True,
                padding="max_length",
                max_length=512
            )

            input_ids = inputs["input_ids"].to(device)
            attention_mask = inputs["attention_mask"].to(device)

            embedding_layer = model.bert.embeddings
            input_embeddings = embedding_layer(input_ids)
            input_embeddings.retain_grad()

            outputs = model(inputs_embeds=input_embeddings, attention_mask=attention_mask)
            logits = outputs.logits
            probs = softmax(logits, dim=-1)
            predicted_class = torch.argmax(probs, dim=-1)

            loss = torch.nn.CrossEntropyLoss()(logits, predicted_class)
            model.zero_grad()
            loss.backward()

            grads = input_embeddings.grad
            token_importances = grads.abs().sum(dim=-1).squeeze().tolist()
            tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist())


            clean_importances = [
                imp for tok, imp in zip(tokens, token_importances)
                if tok not in special_token_set
            ]

            if not clean_importances:
                continue


            n = len(clean_importances)
            region_len = n // 10
            region_importances = []

            for i in range(10):
                start = i * region_len
                end = (i + 1) * region_len if i < 9 else n
                region_score = sum(clean_importances[start:end])
                region_importances.append(region_score)

            # 正規化 (總和為 1)
            total = sum(region_importances)
            if total > 0:
                region_importances = [r / total for r in region_importances]
            else:
                region_importances = [0.0] * 10

            importance_str = "\t".join([f"{r:.4f}" for r in region_importances])
            file.write(f"{gene_name}\t{importance_str}\n")

# ==========================================
# 主程式流程
# ==========================================
# 1. 初始化原始模型資訊
base_repo = 'AIRI-Institute/gena-lm-bert-base-athaliana'
base_model = AutoModel.from_pretrained(base_repo, trust_remote_code=True, revision="main")
gena_module_name = base_model.__class__.__module__
cls = getattr(importlib.import_module(gena_module_name), 'BertForSequenceClassification')

# 2. 載入資料
df = pd.read_table("file/PN_sequence_up_300_downstream_300_seq_Nointron.txt")
df = df[['sequence', 'peak', 'gene']].rename(columns={'sequence': 'text', 'peak': 'label'})
dataset = Dataset.from_pandas(df)

saved_model_dir = "/dicos_ui_home/tzuhsiang/test/organize/saved_models"

for run_id in range(1, 6):
    model_name = f"best_u300d300_L3_model_run{run_id}"
    model_path = os.path.join(saved_model_dir, model_name)

    if not os.path.isdir(model_path):
        print(f"⚠️ 跳過：找不到路徑 {model_path}")
        continue

    print(f"🚀 正在分析模型: {model_name}")

    # 載入權重
    model = cls.from_pretrained(model_path)

    # [修正處]：優先載入本地 Tokenizer，失敗則載入原始倉庫版本
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
    except Exception:
        print(f"ℹ️ 本地無 Tokenizer，改由 {base_repo} 載入...")
        tokenizer = AutoTokenizer.from_pretrained(base_repo)

    # 執行重要性計算
    output_path = f"predictions_u300d300_L3/token_importance_region10_{model_name}.txt"
    calculate_importance(dataset, model, tokenizer, output_path)
    print(f"✅ 完成！檔案已存至: {output_path}")